In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Import required libraries and configure display settings
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import sqlalchemy as sa
from datetime import date

# Disable scientific notation for float display
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Connect to the SQLite database and verify available tables
# ─────────────────────────────────────────────

# Build a relative path to the database file
ruta_datos = Path('..') / 'datos' / 'brutos' / 'ecommerce.db'

# Create a SQLAlchemy engine for the SQLite database
engine = sa.create_engine(f'sqlite:///{ruta_datos}')

# Inspect available tables using SQLAlchemy's inspector
inspector = sa.inspect(engine)
tablas = inspector.get_table_names()
print(f"Tablas disponibles: {tablas}")

# Verify the connection by querying the SQLite master table directly
with engine.connect() as conn:
    resultado = conn.execute(sa.text("SELECT name FROM sqlite_master WHERE type='table';"))
    print("Tablas:", resultado.fetchall())

Tablas disponibles: ['2019-Dec', '2019-Nov', '2019-Oct', '2020-Feb', '2020-Jan']
Tablas: [('2019-Oct',), ('2019-Nov',), ('2019-Dec',), ('2020-Jan',), ('2020-Feb',)]


In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Load each monthly ecommerce table from the SQLite database into separate DataFrames
# ─────────────────────────────────────────────
oct = pd.read_sql('2019-Oct', con=engine)   # October 2019 data
nov = pd.read_sql('2019-Nov', con=engine)   # November 2019 data
dic = pd.read_sql('2019-Dec', con=engine)   # December 2019 data
ene = pd.read_sql('2020-Jan', con=engine)   # January 2020 data
feb = pd.read_sql('2020-Feb', con=engine)   # February 2020 data

In [112]:
print(oct.shape, nov.shape, dic.shape, ene.shape, feb.shape)

(407925, 10) (462833, 10) (351304, 10) (443224, 10) (429790, 10)


In [113]:
print(oct.columns)
print(nov.columns)
print(dic.columns)
print(ene.columns)
print(feb.columns)

Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')
Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')


In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Merge all monthly DataFrames into a single unified dataset
# ─────────────────────────────────────────────
# Concatenate vertically (axis=0); ignore_index resets the index to avoid duplicate labels
df = pd.concat([oct, nov, dic, ene, feb], ignore_index=True, axis=0)
df

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46 UTC,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55 UTC,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50 UTC,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41 UTC,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44 UTC,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5
...,...,...,...,...,...,...,...,...,...,...
2095071,4156660,2020-02-29 23:58:49 UTC,cart,5815662,1487580006317032337,NaN,NaN,0.92,147995998,5ff96629-3627-493e-a25b-5a871ec78c90
2095072,4156663,2020-02-29 23:58:57 UTC,view,5815665,1487580006317032337,NaN,NaN,0.59,147995998,5ff96629-3627-493e-a25b-5a871ec78c90
2095073,4156668,2020-02-29 23:59:05 UTC,cart,5815665,1487580006317032337,NaN,NaN,0.59,147995998,5ff96629-3627-493e-a25b-5a871ec78c90
2095074,4156675,2020-02-29 23:59:28 UTC,view,5817692,1487580010872045658,NaN,NaN,0.79,619841242,18af673b-7fb9-4202-a66d-5c855bc0fd2d


In [115]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2095076 entries, 0 to 2095075
Data columns (total 10 columns):
 #   Column         Dtype  
---  ------         -----  
 0   index          int64  
 1   event_time     str    
 2   event_type     str    
 3   product_id     int64  
 4   category_id    int64  
 5   category_code  str    
 6   brand          str    
 7   price          float64
 8   user_id        int64  
 9   user_session   str    
dtypes: float64(1), int64(4), str(5)
memory usage: 159.8 MB


In [116]:
df.head(3)

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46 UTC,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55 UTC,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50 UTC,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770


In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Parse the event_time column from string to datetime for time-based operations
# ─────────────────────────────────────────────
df['event_time'] = pd.to_datetime(df['event_time'])  # Convert string timestamps to datetime objects
df['event_time'].head()

0   2019-10-01 00:01:46+00:00
1   2019-10-01 00:01:55+00:00
2   2019-10-01 00:02:50+00:00
3   2019-10-01 00:03:41+00:00
4   2019-10-01 00:03:44+00:00
Name: event_time, dtype: datetime64[us, UTC]

In [ ]:
# Drop the auxiliary 'index' column added automatically by SQLite
df = df.drop(columns=['index'])

In [119]:
df.columns

Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Rename columns to Spanish names following the project naming convention
# ─────────────────────────────────────────────
df.columns = [
    'fecha',         # Event timestamp
    'evento',        # Event type (view, cart, remove_from_cart, purchase)
    'producto',      # Product ID
    'categoria',     # Product category path
    'categoria_cod', # Category numeric code
    'marca',         # Product brand
    'precio',        # Product price
    'usuario',       # User ID
    'sesion',        # Session ID
    ]
df

,fecha,evento,producto,categoria,categoria_cod,marca,precio,usuario,sesion
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5
...,...,...,...,...,...,...,...,...,...
2095071,2020-02-29 23:58:49+00:00,cart,5815662,1487580006317032337,NaN,NaN,0.92,147995998,5ff96629-3627-493e-a25b-5a871ec78c90
2095072,2020-02-29 23:58:57+00:00,view,5815665,1487580006317032337,NaN,NaN,0.59,147995998,5ff96629-3627-493e-a25b-5a871ec78c90
2095073,2020-02-29 23:59:05+00:00,cart,5815665,1487580006317032337,NaN,NaN,0.59,147995998,5ff96629-3627-493e-a25b-5a871ec78c90
2095074,2020-02-29 23:59:28+00:00,view,5817692,1487580010872045658,NaN,NaN,0.79,619841242,18af673b-7fb9-4202-a66d-5c855bc0fd2d


In [ ]:
# Check the number of null values per column (data quality assessment)
df.isnull().sum()

fecha                  0
evento                 0
producto               0
categoria              0
categoria_cod    2060411
marca             891646
precio                 0
usuario                0
sesion               506
dtype: int64

In [ ]:
# Drop 'categoria_cod' and 'marca' — not needed for the analysis
df = df.drop(columns=['categoria_cod', 'marca'])

In [ ]:
# Remove all rows containing null values to ensure a clean dataset
df = df.dropna()

In [124]:
df.describe()

,producto,categoria,precio,usuario
count,2094570.00,2094570.00,2094570.00,2094570.00
mean,5487103.56,1553112489392098048.00,8.42,521077545.56
std,1300923.90,167907497920480608.00,19.14,87553855.76
min,3752.00,1487580004807082752.00,-47.62,4661182.00
25%,5724652.00,1487580005754995456.00,2.05,480613387.00
50%,5811665.00,1487580008246412288.00,4.00,553341613.00
75%,5858353.00,1487580013489291520.00,6.86,578406571.00
max,5932595.00,2242903426784559104.00,327.78,622087993.00


In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Quantify rows with zero or negative prices before filtering (data quality check)
# ─────────────────────────────────────────────
df[df.precio <= 0].shape   # Returns (row_count, col_count) for invalid price rows

(20544, 7)

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Remove rows with zero or negative prices to ensure data quality
# ─────────────────────────────────────────────
df = df[df.precio > 0]   # Keep only rows with a valid positive price

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Set the datetime column as the index and extract time-based features for analysis
# ─────────────────────────────────────────────

# Set 'fecha' as the DataFrame index to enable time-series operations
df = df.set_index('fecha')

# Extract temporal granularity features from the DatetimeIndex
df['año'] = df.index.year                           # Calendar year
df['mes'] = df.index.month                          # Month number (1-12)
df['dia'] = df.index.day                            # Day of month (1-31)
df['dia_semana'] = df.index.dayofweek + 1           # Day of week (1=Monday, 7=Sunday)
df['nombre_dia'] = df.index.day_name()              # Day name in English (e.g. "Monday")
df['nombre_mes'] = df.index.month_name()            # Month name in English (e.g. "October")
df['trimestre'] = df.index.quarter                  # Quarter of the year (1-4)
df['semana_año'] = df.index.isocalendar().week      # ISO calendar week number
df['dia_año'] = df.index.dayofyear                  # Day of year (1-366)
df['hora'] = df.index.hour                          # Hour of day (0-23)
df['minuto'] = df.index.minute                      # Minute (0-59)
df['segundo'] = df.index.second                     # Second (0-59)
df['es_fin_semana'] = df.index.dayofweek.isin([5, 6]).astype(int)  # 1 if Saturday or Sunday, else 0
df['es_inicio_mes'] = df.index.is_month_start.astype(int)          # 1 if first day of month, else 0
df['es_fin_mes'] = df.index.is_month_end.astype(int)               # 1 if last day of month, else 0

df.head()

,evento,producto,categoria,precio,usuario,sesion,año,mes,dia,dia_semana,...,nombre_mes,trimestre,semana_año,dia_año,hora,minuto,segundo,es_fin_semana,es_inicio_mes,es_fin_mes
fecha,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019,10,1,2,...,October,4,40,274,0,1,46,0,1,0
2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019,10,1,2,...,October,4,40,274,0,1,55,0,1,0
2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019,10,1,2,...,October,4,40,274,0,2,50,0,1,0
2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019,10,1,2,...,October,4,40,274,0,3,41,0,1,0
2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019,10,1,2,...,October,4,40,274,0,3,44,0,1,0


In [ ]:
# Add a date-only column (no time component) derived from the DatetimeIndex
df['fecha_sin_hora'] = df.index.date

In [ ]:
# Rename the index to 'fecha_hora' to make its meaning explicit
df.index.name = 'fecha_hora'

In [130]:
df.head()

,evento,producto,categoria,precio,usuario,sesion,año,mes,dia,dia_semana,...,trimestre,semana_año,dia_año,hora,minuto,segundo,es_fin_semana,es_inicio_mes,es_fin_mes,fecha_sin_hora
fecha_hora,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019,10,1,2,...,4,40,274,0,1,46,0,1,0,2019-10-01
2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019,10,1,2,...,4,40,274,0,1,55,0,1,0,2019-10-01
2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019,10,1,2,...,4,40,274,0,2,50,0,1,0,2019-10-01
2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019,10,1,2,...,4,40,274,0,3,41,0,1,0,2019-10-01
2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019,10,1,2,...,4,40,274,0,3,44,0,1,0,2019-10-01


In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Load Russian public holidays for 2019-2020 to flag holiday dates in the dataset
# ─────────────────────────────────────────────
import holidays

# Create a dictionary of Russian public holidays for the years covered by the data
f = holidays.RU(years=[2019, 2020])

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Flag rows that fall on a Russian public holiday (binary indicator)
# ─────────────────────────────────────────────

# Apply lookup against the holidays dictionary; returns 1 if the date is a holiday, else 0
df['festivo'] = df['fecha_sin_hora'].apply(lambda x: 1 if x in f else 0)

# Summary statistics for the new feature
print(f"Total de registros en días festivos: {df['festivo'].sum()}")
print(f"Porcentaje de festivos: {df['festivo'].mean() * 100:.2f}%")

Total de registros en días festivos: 133193
Porcentaje de festivos: 6.42%


In [ ]:
# ============================================================
# PURPOSE: Create binary indicators for key ecommerce events in Russia
# Period: Oct 2019 - Feb 2020
# ============================================================

# --- BLACK FRIDAY (Friday after the 4th Thursday of November) ---
black_friday = {date(2019, 11, 29)}

# --- CYBER MONDAY (Monday following Black Friday) ---
cyber_monday = {date(2019, 12, 2)}

# --- SINGLES' DAY (11/11, popular shopping event in Russian ecommerce) ---
dia_soltero = {date(2019, 11, 11)}

# --- WESTERN CHRISTMAS (Dec 24-25, relevant in Russian ecommerce) ---
navidad_occidental = {date(2019, 12, 24), date(2019, 12, 25)}

# --- PRE-NEW YEAR PERIOD (major shopping peak in Russia, Dec 26-31) ---
pre_año_nuevo = {date(2019, 12, 26), date(2019, 12, 27),
                  date(2019, 12, 28), date(2019, 12, 29),
                  date(2019, 12, 30), date(2019, 12, 31)}

# --- RUSSIAN NEW YEAR (official holiday: Jan 1-8) ---
año_nuevo_ruso = {date(2020, 1, d) for d in range(1, 9)}

# --- ORTHODOX CHRISTMAS (January 7th) ---
navidad_ortodoxa = {date(2020, 1, 7)}

# --- VALENTINE'S DAY ---
san_valentin = {date(2020, 2, 14)}

# --- DEFENDER OF THE FATHERLAND DAY (Feb 23, major Russian holiday)
# Feb 22 is included as a pre-holiday shopping day ---
defensor_patria = {date(2020, 2, 22), date(2020, 2, 23)}

# --- APPLY TO THE DATAFRAME ---
# For each special day set, create a binary column: 1 if the date matches, else 0
df['black_friday']       = df['fecha_sin_hora'].apply(lambda x: 1 if x in black_friday else 0)
df['cyber_monday']       = df['fecha_sin_hora'].apply(lambda x: 1 if x in cyber_monday else 0)
df['dia_soltero']        = df['fecha_sin_hora'].apply(lambda x: 1 if x in dia_soltero else 0)
df['navidad_occidental'] = df['fecha_sin_hora'].apply(lambda x: 1 if x in navidad_occidental else 0)
df['pre_año_nuevo']     = df['fecha_sin_hora'].apply(lambda x: 1 if x in pre_año_nuevo else 0)
df['año_nuevo_ruso']    = df['fecha_sin_hora'].apply(lambda x: 1 if x in año_nuevo_ruso else 0)
df['navidad_ortodoxa']   = df['fecha_sin_hora'].apply(lambda x: 1 if x in navidad_ortodoxa else 0)
df['san_valentin']       = df['fecha_sin_hora'].apply(lambda x: 1 if x in san_valentin else 0)
df['defensor_patria']    = df['fecha_sin_hora'].apply(lambda x: 1 if x in defensor_patria else 0)

# --- SUMMARY: count records falling on each special day ---
cols_especiales = ['black_friday', 'cyber_monday', 'dia_soltero', 'navidad_occidental',
                   'pre_año_nuevo', 'año_nuevo_ruso', 'navidad_ortodoxa',
                   'san_valentin', 'defensor_patria']

print("Registros por día especial:")
print(df[cols_especiales].sum().to_string())
print(f"\nTotal registros en algún día especial: {df[cols_especiales].any(axis=1).sum()}")

Registros por día especial:
black_friday          22331
cyber_monday          12069
dia_soltero           13520
navidad_occidental    22099
pre_año_nuevo         39695
año_nuevo_ruso        94222
navidad_ortodoxa      12922
san_valentin          12245
defensor_patria       21026

Total registros en algún día especial: 237207


In [134]:
variables = df.columns.tolist()
variables

['evento',
 'producto',
 'categoria',
 'precio',
 'usuario',
 'sesion',
 'año',
 'mes',
 'dia',
 'dia_semana',
 'nombre_dia',
 'nombre_mes',
 'trimestre',
 'semana_año',
 'dia_año',
 'hora',
 'minuto',
 'segundo',
 'es_fin_semana',
 'es_inicio_mes',
 'es_fin_mes',
 'fecha_sin_hora',
 'festivo',
 'black_friday',
 'cyber_monday',
 'dia_soltero',
 'navidad_occidental',
 'pre_año_nuevo',
 'año_nuevo_ruso',
 'navidad_ortodoxa',
 'san_valentin',
 'defensor_patria']

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Define the priority column order — core business columns placed first
# ─────────────────────────────────────────────
# These columns are the primary identifiers and metrics; they go at the start of the DataFrame
orden = [
    'usuario', 'sesion', 'evento', 'producto', 'precio', 'categoria',
    ]

In [ ]:
# Collect all remaining columns (not in the priority list) and sort them alphabetically
resto = sorted([col for col in variables if col not in orden])

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Reorder DataFrame columns — core business columns first, then the rest alphabetically
# ─────────────────────────────────────────────
df = df[orden + resto]   # Combine priority columns with sorted remaining columns
df.head()

,usuario,sesion,evento,producto,precio,categoria,año,año_nuevo_ruso,black_friday,cyber_monday,...,minuto,navidad_occidental,navidad_ortodoxa,nombre_dia,nombre_mes,pre_año_nuevo,san_valentin,segundo,semana_año,trimestre
fecha_hora,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,view,5843665,9.44,1487580005092295511,2019,0,0,0,...,1,0,0,Tuesday,October,0,0,46,40,4
2019-10-01 00:01:55+00:00,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,cart,5868461,3.57,1487580013069861041,2019,0,0,0,...,1,0,0,Tuesday,October,0,0,55,40,4
2019-10-01 00:02:50+00:00,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,view,5877456,122.22,1487580006300255120,2019,0,0,0,...,2,0,0,Tuesday,October,0,0,50,40,4
2019-10-01 00:03:41+00:00,555448072,b5f72ceb-0730-44de-a932-d16db62390df,view,5649270,6.19,1487580013749338323,2019,0,0,0,...,3,0,0,Tuesday,October,0,0,41,40,4
2019-10-01 00:03:44+00:00,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,view,18082,16.03,1487580005411062629,2019,0,0,0,...,3,0,0,Tuesday,October,0,0,44,40,4


In [ ]:
# Drop auxiliary time columns not needed in the final analytical table
df = df.drop(columns=['nombre_mes', 'dia_año', 'minuto', 'segundo'])

In [ ]:
# ─────────────────────────────────────────────
# PURPOSE: Persist the final processed DataFrame to disk for downstream use
# ─────────────────────────────────────────────
# Save to the intermediate data folder as a pickle file (preserves dtypes and index)
df.to_pickle('../datos/intermedios/tablon_analitico.pickle')